# Task 4 — Amharic predictive text and next-word suggestion

The application (`app/streamlit_app.py`):

1. reads the user's input and normalizes it like the training text;
2. takes the final one or two complete words as the history (trigram model; n is adjustable);
3. retrieves candidate next words: every word seen after that history in GPAC, after its last word,
   and the most frequent words (so an unseen history still gets suggestions); if the user is in the
   middle of a word, only vocabulary words starting with the typed characters;
4. scores every candidate with the interpolated Kneser-Ney probability P(w | history);
5. shows the top five with their probabilities;
6. appends a clicked suggestion to the text and recomputes.

The app loads orders 1–3 of the GPAC count tables memory-mapped from `data/gpac/counts`: the
operating system reads only the table pages that lookups touch, so the app starts quickly and
uses little RAM.

In [1]:
# --- Setup ---------------------------------------------------------------------------
# On Colab / Kaggle, first get this repository and the corpus:
#   !git clone <your-repo-url> nlp-assignment-1
#   %cd nlp-assignment-1
#   !pip install -q wordcloud gdown
#   !gdown 1jEOAORylTBAiNKes_JerD6gFF7cnLFyr -O data/   # GPAC, from the assignment's Drive link
import os, sys
ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, ROOT)
DATA = os.path.join(ROOT, "data")
CORPUS = os.path.join(DATA, "GPAC.txt")     # a .txt file, a directory of .txt files, or a .zip
MODEL_DIR = os.path.join(DATA, "gpac")      # preprocessed splits, vocabulary and counts go here
FIGURES = os.path.join(ROOT, "figures"); os.makedirs(FIGURES, exist_ok=True)
MAX_CHARS = None                            # e.g. 50_000_000 to use only part of the corpus

import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_colwidth", 80)

In [2]:
from ngram import pipeline
vocab, app_counts = pipeline.load_counts(MODEL_DIR, 3)
print({k: app_counts.num_grams(k) for k in range(1, 4)})

{1: 1008337, 2: 21116487, 3: 12634324}


In [3]:
from ngram.generate import Suggester
from ngram.models import KneserNeyModel
from ngram.normalize import preprocess
sug = Suggester(KneserNeyModel(app_counts, 3), vocab)
for text in ["ኢትዮጵያ ውስጥ የቡና", "አዲስ", "የኢትዮጵያ ህዝብ", "በአፍሪካ"]:
    print(text, "->", [(w, round(p, 4)) for w, p in sug.suggest(preprocess(text)[0])])
print("partial word 'ም' after 'የቡና':", sug.suggest(["የቡና"], partial="ም"))

ኢትዮጵያ ውስጥ የቡና -> [('ዋጋ', 0.1044), ('ፍጆታ', 0.0948), ('ድርሻ', 0.0666), ('ግብይት', 0.0492), ('የወጪ', 0.0435)]
አዲስ -> [('አበባ', 0.4728), ('ዘመን', 0.2533), ('አበባን', 0.0271), ('ነገር', 0.0146), ('አድማስ', 0.0137)]
የኢትዮጵያ ህዝብ -> [('አርበኞች', 0.035), ('ነው', 0.019), ('ጋር', 0.014), ('ሆይ', 0.0126), ('ቁጥር', 0.011)]
በአፍሪካ -> [('ህብረት', 0.0781), ('ውስጥ', 0.0617), ('ቀንድ', 0.054), ('ዋንጫ', 0.0409), ('ደረጃ', 0.0323)]
partial word 'ም' after 'የቡና': [('ምርት', 0.051407522294521696), ('ምርትን', 0.0069302104614805396), ('ምርቱ', 0.005141423755692396), ('ምርምር', 0.002009119857933761), ('ምርትና', 0.001827821472691269)]


## Running the app

**Locally:**

```bash
streamlit run app/streamlit_app.py -- --model-dir data/gpac --order 3
```

**On Colab** (Streamlit needs a public tunnel):

```python
!streamlit run app/streamlit_app.py -- --model-dir data/gpac &> streamlit.log &
!npx --yes localtunnel --port 8501   # open the printed URL; the password is this machine's IP:
!curl -s ipv4.icanhazip.com
```

**Terminal version:** `python app/cli.py --model-dir data/gpac`.